# 06. nn.Module Internals, Weight Parametrizations & Meta Devices

### Historical Lineage & Theoretical Foundations
- **Lineage:** *Collobert et al. (2011, Torch7); Paszke et al. (2019, PyTorch); Lezcano-Casado & Martínez-Rubio (2019, Orthogonal Parametrization)*.
- **Core Purpose:** State partitioning (`Parameter` vs persistent/non-persistent buffers), manifold parameterization ($SO(n)$ via Cayley transforms), and zero-memory dry-run shape tracing via `meta` device.

### Decision Boundary: When NOT to Use
> [!WARNING]
> - **DO NOT USE in-place weight mutation in forward:** Overwriting `.weight` directly breaks autograd graphs and invalidates optimizer momentum; use `torch.nn.utils.parametrize`.
> - **DO NOT USE persistent buffers for runtime caches:** Dynamic buffers bloat checkpoints and corrupt saved weights; use `register_buffer(..., persistent=False)`.
> - **DO NOT allocate real device memory for dry-run shapes:** Sizing 70B+ LLMs directly on GPU/CPU causes immediate OOM; use `torch.device("meta")`.


### Mathematical Formulation & Asymptotic Complexity

Given unconstrained matrix $\mathbf{M} \in \mathbb{R}^{d \times d}$, project onto Lie algebra $\mathfrak{so}(d)$:
$$\mathbf{A} = \text{triu}(\mathbf{M}, 1) - \text{triu}(\mathbf{M}, 1)^\top \implies \mathbf{A}^\top = -\mathbf{A}$$

The **Cayley Transform** maps $\mathbf{A}$ to the special orthogonal Lie group $SO(d)$:
$$\mathbf{W} = (\mathbf{I} - \mathbf{A})(\mathbf{I} + \mathbf{A})^{-1}$$

Because $\mathbf{W}^\top \mathbf{W} = \mathbf{I}$, the linear map preserves $\ell_2$ norm identically:
$$\|\mathbf{W} \mathbf{x}\|_2^2 = \mathbf{x}^\top \mathbf{W}^\top \mathbf{W} \mathbf{x} = \|\mathbf{x}\|_2^2$$

**Asymptotic Complexity:**
- Cayley Transform: Time $\mathcal{O}(d^3)$ matrix inversion; Space $\mathcal{O}(d^2)$.
- Forward Projection: Time $\mathcal{O}(B \cdot d^2)$; Activation Memory $\mathcal{O}(B \cdot d)$.
- Meta Device Trace: Time $\mathcal{O}(1)$ FLOPs; Physical Allocation: **0 Bytes**.


In [ ]:
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.utils.parametrize as parametrize

import core

env = core.setup_env(seed=42)

ckpt_path = Path("weights/00_foundations/06_nn_module_internals_parametrizations_and_meta_device.safetensors")
if not ckpt_path.is_file():
    ckpt_path = Path("../../weights/00_foundations/06_nn_module_internals_parametrizations_and_meta_device.safetensors")
weights, test_in, golden_out, meta = core.parity.load_reference_checkpoint(ckpt_path)
dim = int(meta.get("dim", 3))
print(f"Device: {env.device} | Dim: {dim} | Checkpoint: {ckpt_path.name}")


In [ ]:
class CayleyOrthogonal(nn.Module):
    def __init__(self, dim: int) -> None:
        super().__init__()
        self.register_buffer("eye", torch.eye(dim, dtype=torch.float32), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        tri_u = torch.triu(x, diagonal=1)
        skew_a = tri_u - tri_u.T
        inv_term = torch.linalg.inv(self.eye + skew_a)
        return (self.eye - skew_a) @ inv_term

class OrthogonalLinear(nn.Module):
    def __init__(self, dim: int) -> None:
        super().__init__()
        self.linear = nn.Linear(dim, dim, bias=True)
        parametrize.register_parametrization(self.linear, "weight", CayleyOrthogonal(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x @ self.linear.weight + self.linear.bias

model = OrthogonalLinear(dim=dim)
assert parametrize.is_parametrized(model.linear, "weight")
print("OrthogonalLinear initialized with Cayley parametrization.")


In [ ]:
with torch.no_grad():
    model.linear.parametrizations.weight.original.copy_(weights["skew_param"])
    model.linear.bias.copy_(weights["bias"])

# 1. Invariant: Orthogonality & Isometry
w_eff = model.linear.weight
torch.testing.assert_close(w_eff.T @ w_eff, torch.eye(dim), atol=1e-6, rtol=1e-6)
torch.testing.assert_close(torch.norm(test_in, dim=-1), torch.norm(test_in @ w_eff, dim=-1), atol=1e-5, rtol=1e-5)

# 2. Transitive Parity vs Golden Checkpoint
x_act = test_in.clone().detach().requires_grad_(True)
pred = model(x_act)
torch.testing.assert_close(pred, golden_out, atol=1e-5, rtol=1e-5)

loss = 0.5 * torch.sum((pred - weights["target"]) ** 2)
loss.backward()
torch.testing.assert_close(model.linear.parametrizations.weight.original.grad, weights["grad_skew"], atol=1e-5, rtol=1e-5)
torch.testing.assert_close(model.linear.bias.grad, weights["grad_bias"], atol=1e-5, rtol=1e-5)
torch.testing.assert_close(x_act.grad, weights["grad_input"], atol=1e-5, rtol=1e-5)
print("All mathematical invariant checks (Orthogonality + Gradient Parity) passed successfully.")


In [ ]:
# Micro-benchmark: Unconstrained Linear vs Cayley Orthogonal Linear
plain = nn.Linear(dim, dim)
def run_plain():
    x = test_in.clone().detach().requires_grad_(True)
    loss = 0.5 * torch.sum((plain(x) - weights["target"]) ** 2)
    loss.backward()

def run_ortho():
    x = test_in.clone().detach().requires_grad_(True)
    loss = 0.5 * torch.sum((model(x) - weights["target"]) ** 2)
    loss.backward()

track_fns = {"Unconstrained Linear": run_plain, "Cayley Orthogonal Linear": run_ortho}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=test_in.shape[0])
print(core.benchmark.format_benchmark_table(bench_results))


In [ ]:
# [!WARNING] PATHOLOGICAL MISUSE & FAILURE MODE: Meta Device Data Access
# 1. Instantiating deep architecture with zero physical RAM
with torch.device("meta"):
    deep_meta = nn.Sequential(*[nn.Linear(1024, 1024) for _ in range(50)])
meta_x = torch.randn(8, 1024, device="meta")
meta_out = deep_meta(meta_x)
assert meta_out.is_meta and meta_out.shape == (8, 1024)

# 2. Catch illegal attempt to read data from meta device
try:
    _ = meta_out[0, 0].item()
    raise AssertionError("Meta tensor should not permit item() access!")
except RuntimeError as e:
    print(f"[FAILURE DEMONSTRATED] Meta tensor correctly rejected physical read: {e}")

# 3. CORRECT PATTERN: Verify torch.compile & torch.export on materialized model
ok_exp, msg_exp = core.export.verify_torch_export(plain, test_in)
ok_comp, msg_comp = core.export.verify_torch_compile_fullgraph(plain, test_in)
assert ok_exp and ok_comp, "Compilation / export failed"
print("[CORRECT] Model verified with torch.export and torch.compile(fullgraph=True).")


## Architectural Topology, Key Takeaways & Curriculum Navigation

### Module Internals & Parametrization Topology
<svg width="650" height="170" xmlns="http://www.w3.org/2000/svg" style="background:#0f172a; border-radius:8px; font-family:system-ui, sans-serif;">
  <defs>
    <marker id="arrow-p" viewBox="0 0 10 10" refX="6" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M 0 1 L 10 5 L 0 9 z" fill="#38bdf8"/>
    </marker>
  </defs>
  <text x="30" y="25" fill="#94a3b8" font-size="12" font-weight="bold">MODULE INTERNALS, PARAMETRIZATIONS &amp; META DEVICE</text>
  <rect x="30" y="50" width="130" height="35" rx="6" fill="#1e293b" stroke="#38bdf8" stroke-width="1.5"/>
  <text x="95" y="72" fill="#38bdf8" font-size="11" text-anchor="middle">Parameter M in R^{dxd}</text>
  <line x1="160" y1="67" x2="220" y2="67" stroke="#38bdf8" stroke-width="2" marker-end="url(#arrow-p)"/>
  <rect x="220" y="50" width="180" height="35" rx="6" fill="#1e293b" stroke="#10b981" stroke-width="1.5"/>
  <text x="310" y="72" fill="#10b981" font-size="11" text-anchor="middle">Cayley: W = (I-A)(I+A)^-1</text>
  <line x1="400" y1="67" x2="460" y2="67" stroke="#38bdf8" stroke-width="2" marker-end="url(#arrow-p)"/>
  <rect x="460" y="50" width="140" height="35" rx="6" fill="#1e293b" stroke="#38bdf8" stroke-width="1.5"/>
  <text x="530" y="72" fill="#38bdf8" font-size="11" text-anchor="middle">Isometric Map: Y = XW+b</text>
  <rect x="30" y="110" width="280" height="35" rx="6" fill="#0f172a" stroke="#f59e0b" stroke-width="1.5" stroke-dasharray="4 4"/>
  <text x="170" y="132" fill="#f59e0b" font-size="11" text-anchor="middle">torch.device('meta'): 0-Byte Dry Run</text>
</svg>

### Key Takeaways
1. **Parametrization Protocol:** `torch.nn.utils.parametrize` cleanly maps unconstrained Euclidean parameters onto geometric manifolds ($SO(n)$).
2. **Cayley Invariance:** Eigenvalues of $I + A$ have real part 1, guaranteeing unconditional invertibility and exact isometry ($\|Wx\|_2 = \|x\|_2$).
3. **Zero-Byte Staging:** Use `torch.device("meta")` for dry-run shape propagation, sizing, and sharding planning without memory allocation.

---
**Curriculum Navigation:**  
&larr; [Previous: 05. Hooks & Saved Tensor Offloading](05_hooks_and_saved_tensor_hooks.ipynb) | [Master Index: README.md](../../README.md) | Next: 07. Feature Engineering, Scalers & Tabular Preprocessing (Upcoming) &rarr;
